# VisionGuard exploration

Walk the stack without a GPU: taxonomy, synthetic frames, dummy inference, tracking, fusion, and overlay.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root / "src"))

from visionguard.capture import synthetic_frame
from visionguard.hazards.taxonomy import default_taxonomy
from visionguard.pipeline import RealTimePipeline

tax = default_taxonomy()
print(f"{len(tax.classes)} classes")
for spec in tax.classes[:6]:
    print(f"  {spec.class_id:2d}  {spec.detector_name:16s}  {spec.severity.value}")

In [ ]:
from matplotlib import pyplot as plt
import cv2

pipe = RealTimePipeline.from_files(backend="dummy", zones_path=str(root / "configs/zones.example.yaml"))
frame = synthetic_frame(960, 540, seed=7)
result = pipe.process_frame(frame)
hud = pipe.annotate(result)
print("detections", [d.class_name for d in result.detections])
print("risk", round(result.risk_score, 2), "ms", round(result.inference_ms, 2))

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(hud, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("VisionGuard overlay")
plt.show()

In [ ]:
risks = []
for i in range(20):
    r = pipe.process_frame(synthetic_frame(640, 384, seed=i))
    risks.append(r.risk_score)
plt.figure(figsize=(8, 3))
plt.plot(risks, color="#00e5a8")
plt.title("Fused risk EMA")
plt.xlabel("frame")
plt.ylabel("risk")
plt.show()
print(pipe.telemetry.snapshot())